# Seeking Alpha Extractor

This notebook uses a real Chromium window through Playwright.

- It can extract content that your browser session can actually see.
- It does not bypass paywalls, CAPTCHAs, or login controls.
- Seeking Alpha is more reliable in headed mode than headless mode.
- The intended flow is: open browser, log in manually if needed, then capture page content to JSON.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "playwright": "playwright",
    "pandas": "pandas",
}
missing_packages = [pkg for module_name, pkg in required_packages.items() if importlib.util.find_spec(module_name) is None]

if missing_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing_packages], check=True)

subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)

: 

In [ ]:
from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not find the repo root from this notebook.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from src.utils.seeking_alpha_browser import SeekingAlphaBrowser, save_snapshot, snapshot_to_text

REPO_ROOT

In [ ]:
import os

PROFILE_DIR = REPO_ROOT / "data" / "browser_profiles" / "seeking_alpha"
OUTPUT_DIR = REPO_ROOT / "data" / "seeking_alpha" / "captures"
PROFILE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

targets_env = os.getenv("SA_TARGETS", "")
TARGETS = [item.strip() for item in targets_env.split(",") if item.strip()] or [
    "AAPL",
    # "https://seekingalpha.com/article/1234567-example-article",
]

HEADLESS = os.getenv("SA_HEADLESS", "0") == "1"
WAIT_FOR_LOGIN = os.getenv("SA_WAIT_FOR_LOGIN", "1") == "1"
SLOW_MO_MS = int(os.getenv("SA_SLOW_MO_MS", "150"))
SCROLL_STEPS = int(os.getenv("SA_SCROLL_STEPS", "10"))

PROFILE_DIR, OUTPUT_DIR

In [ ]:
browser = await SeekingAlphaBrowser(
    user_data_dir=PROFILE_DIR,
    headless=HEADLESS,
    slow_mo_ms=SLOW_MO_MS,
).start()

page = await browser.open("https://seekingalpha.com", scroll=False)
await browser.wait_for_user(
    "The browser window is open. Sign in to Seeking Alpha or dismiss banners if needed, then press Enter here.",
    enabled=WAIT_FOR_LOGIN,
)

page.url

In [ ]:
results = []
saved_files = []

for target in TARGETS:
    page = await browser.open(target, scroll=True, scroll_steps=SCROLL_STEPS)
    snapshot = await browser.extract_snapshot(page)
    saved_path = save_snapshot(snapshot, OUTPUT_DIR)
    results.append(snapshot)
    saved_files.append(saved_path)

print(f"Captured {len(results)} page(s)")
saved_files

In [ ]:
summary_rows = []

for snapshot in results:
    facts = snapshot["page_facts"]
    summary_rows.append(
        {
            "url": snapshot["url"],
            "title": facts.get("title"),
            "author": facts.get("author"),
            "published_at": facts.get("published_at"),
            "headings": len(snapshot["headings"]),
            "text_blocks": len(snapshot["text_blocks"]),
            "tables": len(snapshot["tables"]),
            "links": len(snapshot["links"]),
        }
    )

pd.DataFrame(summary_rows)

In [ ]:
if not results:
    raise RuntimeError("Run the capture cell first.")

preview_index = 0
print(snapshot_to_text(results[preview_index], max_text_blocks=15))

In [ ]:
results[0]["tables"][:1] if results else []

In [ ]:
if "browser" in globals():
    await browser.close()